In [2]:
import pandas as pd


sms_db = pd.read_csv("./all_sms.csv")
sms_db['date'] = pd.to_datetime(sms_db['date'], unit='s')
sms_db.dtypes

id                     int64
date          datetime64[us]
sender                   str
text                     str
is_from_me              bool
service                  str
dtype: object

In [13]:
sms_db = sms_db[sms_db['sender'] != 'me']
sms_db

,id,date,sender,text,is_from_me,service
2,3,2021-05-21 07:07:00.000000,VD-HDFCBK,Your mobile number and device is successfully ...,False,SMS
3,4,2021-05-21 07:07:03.000000,VK-HDFCBK,Your mobile number and device is successfully ...,False,SMS
5,6,2021-05-21 07:09:37.000000,VK-HDFCBK,Your mobile number and device is successfully ...,False,SMS
7,8,2021-05-21 07:09:40.000000,VK-HDFCBK,Device SMS count Exceeded For The Day,False,SMS
10,11,2021-05-21 08:19:47.000000,VK-ICICIB,"Dear Customer, registration for UPI has starte...",False,SMS
...,...,...,...,...,...,...
12124,18204,2026-03-04 14:26:16.869241,TRAIND-G,"ವಂಚಕರ ವಿರುದ್ಧ ಸ್ಪ್ಯಾಮ್ ವರದಿ ಮಾಡಿ. ಆದರೆ, ಫೋನಿನ ...",False,SMS
12125,18212,2026-03-05 07:24:09.798212,TRAIND-G,स्पॅमरच्या विरोधात कारवाईसाठी स्पॅमचा अहवाल द्...,False,SMS
12126,18215,2026-03-05 08:04:54.587784,AIRTEL-S,Never respond to emails/embedded links in mess...,False,SMS
12127,18223,2026-03-05 22:34:08.041849,MAHABK-S,"Dear Customer, there is no transaction in your...",False,SMS


In [14]:
# Check whether `id` is sequential and identify gaps/duplicates
id_series = pd.to_numeric(sms_db["id"], errors="coerce").dropna().astype(int)

row_count = len(sms_db)
min_id = id_series.min()
max_id = id_series.max()
unique_ids = id_series.nunique()

is_exact_1_to_n = (min_id == 1) and (max_id == row_count) and (unique_ids == row_count)
is_contiguous_min_to_max = (unique_ids == (max_id - min_id + 1))

print(f"rows: {row_count}")
print(f"id min/max: {min_id}/{max_id}")
print(f"unique ids: {unique_ids}")
print(f"sequential 1..rows? {is_exact_1_to_n}")
print(f"contiguous min..max? {is_contiguous_min_to_max}")

# Missing IDs in the observed min..max range
expected_ids = pd.Index(range(min_id, max_id + 1))
missing_ids = expected_ids.difference(pd.Index(id_series.unique()))

print(f"missing id count: {len(missing_ids)}")
print("first 20 missing ids:", missing_ids[:20].tolist())

# Duplicate IDs (if any)
dup_counts = id_series.value_counts()
duplicate_ids = dup_counts[dup_counts > 1]
print(f"duplicate id count: {len(duplicate_ids)}")
if len(duplicate_ids) > 0:
    print("first 20 duplicate ids:", duplicate_ids.head(20).to_dict())

rows: 11984
id min/max: 3/18252
unique ids: 11984
sequential 1..rows? False
contiguous min..max? False
missing id count: 6266
first 20 missing ids: [5, 7, 9, 10, 14, 15, 26, 35, 36, 37, 38, 39, 40, 50, 60, 66, 79, 84, 99, 137]
duplicate id count: 0


In [15]:
sms_db['sender'].value_counts()

sender
VZ-611123          1007
VZ-612345          1001
VD-611126           256
AD-HDFCBK           242
JZ-JioPay           207
                   ... 
ADHAAR-S(smsft)       1
+918977520171         1
HDFCBN-S(smsft)       1
SBIBNK-S(smsft)       1
AIRTEL-S              1
Name: count, Length: 1648, dtype: int64

In [3]:
# Filter out sent messages
inbox_df = sms_db[sms_db['sender'] != 'me'].copy()

# Basic sender stats
unique_senders = inbox_df['sender'].nunique()
print(f"Total incoming messages: {len(inbox_df)}")
print(f"Total unique senders: {unique_senders}")

# Top 20 senders by volume
top_senders = inbox_df['sender'].value_counts().head(50)
print("\nTop 20 senders by message volume:")
print(top_senders)

Total incoming messages: 11984
Total unique senders: 1648

Top 20 senders by message volume:
sender
VZ-611123    1007
VZ-612345    1001
VD-611126     256
AD-HDFCBK     242
JZ-JioPay     207
VM-611121     192
VM-HDFCBK     163
VZ-EXPIRY     146
JK-620016     146
JD-620016     136
JM-620016     134
VD-HDFCBN     133
JZ-JIOINF     132
JZ-JIOPAY     125
JX-HDFCBK     121
JX-620016     119
VM-HDFCBN     116
VM-ViCARE     114
VD-HDFCBK     111
VM-611126     109
VM-IDFCFB     101
5512111       100
TX-SLCEIT      96
VZ-ViRoam      81
QP-CHGGIn      71
JZ-JioSvc      70
VK-NSETRA      66
VM-NSESMS      63
CP-OneCrd      62
AX-HDFCBK      59
BW-SBIUPI      58
AD-HDFCBN      57
VM-FLPKRT      56
TM-IDFCFB      56
5040406        55
TM-TATANU      55
JM-HDFCBN      54
JZ-JIOVOC      53
VM-adidas      52
50404          51
BH-CHGGIn      49
BA-SBIUPI      49
VD-IDFCFB      49
JD-SBIUPI      47
VM-BSELTD      45
AD-OneCrd      43
AD-IDFCFB      43
AD-SBIUPI      40
VM-NOBRKR      40
VK-MAHABK      39


In [4]:
import re

def categorize_sender(sender):
    sender = str(sender).strip()
    
    # 1. Regular Mobile Number: optional '+' followed by 10 to 15 digits
    if re.match(r'^\+?[0-9]{10,15}$', sender):
        return 'Mobile Number'
    
    # 2. Numeric Shortcode: purely numeric, less than 10 digits
    elif re.match(r'^[0-9]{3,9}$', sender):
        return 'Numeric Shortcode'
    
    # 3. Commercial/Brand: standard XX-YYYYYY Indian gateway format
    elif re.match(r'^[A-Za-z]{2}-.+$', sender):
        return 'Commercial/Brand (Prefixed)'
    
    # 4. Commercial/Brand: other textual names (e.g., 'JioPay', 'HDFCBK' without prefix)
    elif re.search(r'[A-Za-z]', sender):
        return 'Commercial/Brand (Textual)'
    
    # 5. Others: to catch edge cases
    else:
        return 'Others'

# Apply the categorization
inbox_df['sender_category'] = inbox_df['sender'].apply(categorize_sender)

# Get counts
category_counts = inbox_df['sender_category'].value_counts()
print("Message volume by Sender Category:")
print(category_counts, "\n")

# Let's inspect unique senders in other/edge-case categories
others_senders = inbox_df[inbox_df['sender_category'] == 'Others']['sender'].unique()
print(f"Unique senders in 'Others': {len(others_senders)}")
if len(others_senders) > 0:
    print(others_senders[:20])

Message volume by Sender Category:
sender_category
Commercial/Brand (Prefixed)    11259
Mobile Number                    325
Numeric Shortcode                268
Commercial/Brand (Textual)       132
Name: count, dtype: int64 

Unique senders in 'Others': 0


In [18]:
inbox_df[inbox_df['sender_category'] == 'Mobile Number']

,id,date,sender,text,is_from_me,service,sender_category
50,55,2021-05-29 10:00:11.000000,+919351197465,6-Layers Protection C0R0NA-virus.\nwear N95 Ma...,False,SMS,Mobile Number
94,102,2021-06-06 07:17:53.000000,+917703967847,"Dear 96991837XX,\n\nRs.5500 in Your Rummy Acco...",False,SMS,Mobile Number
118,126,2021-06-09 10:02:53.000000,+918700855621,"Congratulations,\n\nRs.5500 in Your Rummy Acco...",False,SMS,Mobile Number
266,269,2021-06-30 05:10:44.000000,+917990025887,Your 9699XXXX27 is selected for Rs 5250.00 dir...,False,SMS,Mobile Number
398,409,2021-07-26 07:06:58.000000,+918595950300,Your Citi Credit Card No. YCPTXXXXXXXX8795 has...,False,SMS,Mobile Number
...,...,...,...,...,...,...,...
12112,17882,2026-02-07 20:33:24.276616,+919981657790,😞,True,SMS,Mobile Number
12113,17883,2026-02-07 20:33:25.558564,+919981657790,😞,True,SMS,Mobile Number
12114,17884,2026-02-07 20:33:47.968934,+919981657790,Tum so hi rhi ho na,True,SMS,Mobile Number
12115,17885,2026-02-07 20:34:13.965093,+919981657790,Kitna piya tha itna,True,SMS,Mobile Number


In [5]:
import re
import pandas as pd
import sys

# 1. Focus only on Commercial and Shortcode messages (Drop personal mobile numbers)
commercial_df = inbox_df[inbox_df['sender_category'] != 'Mobile Number'].copy()

# ==========================================
# STAGE 1: AMOUNT REQUIREMENT 
# ==========================================
# Transactions almost strictly require a currency format + number. 
# We're making sure we capture varied spaces between currency like 'Rs. 500', 'Rs.500', 'INR500', '₹ 100'.
amount_pattern = re.compile(
    r'(?:rs\.?|inr|₹)\s*[\d,]+(?:\.\d{1,2})?|[\d,]+(?:\.\d{1,2})?\s*(?:rs\.?|inr|₹)', 
    re.IGNORECASE
)

def has_amount(text):
    if not isinstance(text, str):
        return False
    return bool(amount_pattern.search(text))

print("STAGE 1: Applying amount requirement filter...")
commercial_df['1_has_amount'] = commercial_df['text'].apply(has_amount)

# Analyze breakdown
txn_counts_1 = commercial_df['1_has_amount'].value_counts()
print(f"\nTotal Commercial/Shortcode messages: {len(commercial_df)}")
print("Messages with Amount vs Without (Likely Transaction vs Not):")
print(txn_counts_1)
print(f"Percentage retaining amount: {(txn_counts_1.get(True, 0) / len(commercial_df)) * 100:.2f}%\n")

# Sample match
print("--- SAMPLE MATCHED MESSAGES WITH AMOUNT (True) ---")
matched_sample = commercial_df[commercial_df['1_has_amount'] == True]['text'].dropna().sample(5, random_state=42)
for text in matched_sample:
    print(f"- {text[:150].replace(chr(10), ' ')}")

# Sample unmatch
print("\n--- SAMPLE FILTERED OUT AS NO AMOUNT (False) ---")
unmatched_sample = commercial_df[commercial_df['1_has_amount'] == False]['text'].dropna().sample(5, random_state=42)
for text in unmatched_sample:
    print(f"- {text[:150].replace(chr(10), ' ')}")

STAGE 1: Applying amount requirement filter...

Total Commercial/Shortcode messages: 11659
Messages with Amount vs Without (Likely Transaction vs Not):
1_has_amount
False    6480
True     5179
Name: count, dtype: int64
Percentage retaining amount: 44.42%

--- SAMPLE MATCHED MESSAGES WITH AMOUNT (True) ---
- Vi T20 Cricket Pack! Rs901=3GB/day + 48GB EXTRA + All India UL Calls valid for 70days + 1year Disney+ Hotstar Mobile. Recharge NOW! https://bit.ly/3io
- Rs500.0 debited@SBI UPI frm A/cX6254 on 26Aug22 RefNo 223838987367. If not done by u, fwd this SMS to 9223008333/Call 1800111109 or 09449112211 to blo
- Dear SBI User, your A/c X6254-debited by Rs838.0 on 07Jun22 transfer to Flipkart Ref No 215810058234. If not done by u, fwd this SMS to 9223008333/Cal
- Dear Manish,  Get more cash effortlessly with HDFC Bank Personal Loan up to Rs 5 lacs at Best rates. Paperless. Click: hdfcbk.net/vXKVm5vb  T&C
- Missing your favorite songs? Enjoy 3months Ad Free music in HD quality with 6GB for 15

In [ ]:
import os

# Create RESULTS directory
os.makedirs('RESULTS', exist_ok=True)

# Separate the datasets based on Stage 1
likely_transactions = commercial_df[commercial_df['1_has_amount'] == True]
non_transactions = commercial_df[commercial_df['1_has_amount'] == False]

# Save to a single Excel file with two sheets
output_path = 'RESULTS/1_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 6624 likely transactions and 5035 non-transactions to RESULTS/1_transactions_split.xlsx


In [6]:
# ==========================================
# STAGE 2: OTP EXCLUSION
# ==========================================
# Remove messages containing OTP keywords. Many auth messages have amounts (like "OTP to pay Rs. 500") 
# but are NOT the final transaction receipt we're looking for.

otp_pattern = re.compile(
    r'\b(otp|one time password|verification code|pin|login code|do not share)\b', 
    re.IGNORECASE
)

def not_is_otp(row):
    text = row['text']
    if not isinstance(text, str):
        return False
        
    # MUST have passed Stage 1 (has amount)
    if not row['1_has_amount']:
        return False
        
    # MUST NOT contain common OTP/Auth phrases
    return not bool(otp_pattern.search(text))

print("STAGE 2: Applying OTP exclusion filter on remaining messages...")
commercial_df['2_not_is_otp'] = commercial_df.apply(not_is_otp, axis=1)

# Analyze refined breakdown
txn_counts_2 = commercial_df['2_not_is_otp'].value_counts()
print("\nRefined classification after OTP exclusion (Likely Transaction vs Not):")
print(txn_counts_2)
print(f"Percentage classified as actual transaction: {(txn_counts_2.get(True, 0) / len(commercial_df)) * 100:.2f}%\n")

# Sample match
print("--- SAMPLE REFINED TRANSACTIONS NON-OTP (True) ---")
refined_true = commercial_df[commercial_df['2_not_is_otp'] == True]['text'].dropna().sample(min(5, txn_counts_2.get(True, 0)), random_state=42)
for text in refined_true:
    print(f"- {text[:150].replace(chr(10), ' ')}")

# Sample unmatch
print("\n--- SAMPLE FILTERED OUT AS OTP (Passed Stage 1, but rejected in Stage 2) ---")
filtered_out = commercial_df[(commercial_df['1_has_amount'] == True) & (commercial_df['2_not_is_otp'] == False)]['text'].dropna()
if len(filtered_out) > 0:
    for text in filtered_out.sample(min(5, len(filtered_out)), random_state=42):
        print(f"- {text[:150].replace(chr(10), ' ')}")
else:
    print("No messages were filtered out in this stage.")

STAGE 2: Applying OTP exclusion filter on remaining messages...

Refined classification after OTP exclusion (Likely Transaction vs Not):
2_not_is_otp
False    6511
True     5148
Name: count, dtype: int64
Percentage classified as actual transaction: 44.15%

--- SAMPLE REFINED TRANSACTIONS NON-OTP (True) ---
- Plan expired! Recharge now on PhonePe & get rewards upto Rs.400 each on first 3 recharges for self & family. Recharge with Jio no.8830529924 with Rs.6
- HDFC Bank: Rs 15000.00 debited from A/c XXXXXX9141 towards TATA TECHNOLOGIES LIMITED on 29/11/23. Application No. 333302719804, UMN: 0a637daf4ced1e17e
- Winner! You have won Rs.8850 welcome bonus on Junglee Rummy. Download the app now to claim: http://1kx.in/XoxGMwudiIP
- Dear Customer, Amt Due Rs.7055 on HDFC Bank Card X0816? Pay with PayZapp, your safe Bank zone. Steps: PayZapp-Bill-Credit Card Pay Now: hdfcbk.io/k/DU
- Client: X2YPQ, Rs.7284.72 is total amount payable to you at the end of date 03/06/24. This includes all trades 

In [ ]:
import os

# Separate the datasets based on Stage 2
likely_transactions = commercial_df[commercial_df['2_not_is_otp'] == True]
non_transactions = commercial_df[commercial_df['2_not_is_otp'] == False]

output_path = 'RESULTS/2_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 4386 likely transactions and 7273 non-transactions to RESULTS/2_transactions_split.xlsx


In [7]:
# ==========================================
# STAGE 3: PROMO EXCLUSION
# ==========================================
# Some promotional messages contain amounts (e.g. "Get Rs 500 cashback!")
# We use this stage to filter those out.

promo_pattern = re.compile(
    r'\b(offer|offers|discount|promo|code|flat|off|sale|win|free|cashback|apply|starting @|subscribe|bonus|gift|hurry|rewards?)\b|%', 
    re.IGNORECASE
)

def not_is_promo(row):
    text = row['text']
    if not isinstance(text, str):
        return False
    
    # MUST have passed Stage 2 (has amount and not an OTP)
    if not row['2_not_is_otp']:
        return False
        
    # MUST NOT look like an obvious promotion
    return not bool(promo_pattern.search(text))

print("STAGE 3: Applying tightened filtering to remove promotional noise...")
commercial_df['3_not_is_promo'] = commercial_df.apply(not_is_promo, axis=1)

# Analyze refined breakdown
txn_counts_3 = commercial_df['3_not_is_promo'].value_counts()
print("\nRefined classification after Promo exclusion (Likely Transaction vs Not):")
print(txn_counts_3)
print(f"Percentage classified as actual transaction: {(txn_counts_3.get(True, 0) / len(commercial_df)) * 100:.2f}%\n")

print("--- SAMPLE REFINED TRANSACTIONS NON-PROMO (True) ---")
refined_true = commercial_df[commercial_df['3_not_is_promo'] == True]['text'].dropna().sample(min(5, txn_counts_3.get(True, 0)), random_state=42)
for text in refined_true:
    print(f"- {text[:150].replace(chr(10), ' ')}")

print("\n--- SAMPLE FILTERED OUT AS PROMO (Passed Stage 2, but rejected here) ---")
filtered_out = commercial_df[(commercial_df['2_not_is_otp'] == True) & (commercial_df['3_not_is_promo'] == False)]['text'].dropna()
if len(filtered_out) > 0:
    for text in filtered_out.sample(min(5, len(filtered_out)), random_state=42):
        print(f"- {text[:150].replace(chr(10), ' ')}")
else:
    print("No messages were filtered out in this stage.")

STAGE 3: Applying tightened filtering to remove promotional noise...

Refined classification after Promo exclusion (Likely Transaction vs Not):
3_not_is_promo
False    8169
True     3490
Name: count, dtype: int64
Percentage classified as actual transaction: 29.93%

--- SAMPLE REFINED TRANSACTIONS NON-PROMO (True) ---
- Dear SBI UPI User, ur A/cX6254 credited by Rs3 on 03Feb22 by  (Ref no 203485402422)
- Rs.2326.77 spent on HDFC Bank Card x6719 at Raz*Simpl on 2024-07-17:07:55:49.Not U? To Block & Reissue Call 18002586161/SMS BLOCK CC 6719 to 730808080
- Client: X2YPQ, Rs.4393.59 is total amount payable to you at the end of date 11/09/24. This includes all trades till 11/09/24. - Kotak Securities
- Rs.209 ने रिचार्ज करा आणि 1 जीबी डेटा / दिवस, अमर्यादित व्हॉइस कॉल्स  मिळवा, वैधता 28 दिवस. www.jio.com/r/v1DBOaNVl क्लिक करा
- अभी A23 डाउनलोड करें और Rs.10,000 का बोनस पाये r09.in/s/cBw0xuFZur

--- SAMPLE FILTERED OUT AS PROMO (Passed Stage 2, but rejected here) ---
- Refer your friends & a

In [ ]:
import os

# Separate the datasets based on Stage 3
likely_transactions = commercial_df[commercial_df['3_not_is_promo'] == True]
non_transactions = commercial_df[commercial_df['3_not_is_promo'] == False]

output_path = 'RESULTS/3_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 3485 likely transactions and 8174 non-transactions to RESULTS/3_transactions_split.xlsx


In [10]:
# ==========================================
# STAGE 4: REQUEST EXCLUSION
# ==========================================
# Remove messages containing words like 'request' or 'requested'. 
# A transaction alert shouldn't typically be a request for money, but a confirmed action.

request_pattern = re.compile(
    r'\b(request|requested|requesting|requests)\b', 
    re.IGNORECASE
)

def not_is_request(row):
    text = row['text']
    if not isinstance(text, str):
        return False
        
    # MUST have passed Stage 3 (Amount, No OTP, No Promo)
    if not row['3_not_is_promo']:
        return False
        
    # MUST NOT contain request keywords
    return not bool(request_pattern.search(text))

print("STAGE 4: Applying requested exclusion filter on remaining messages...")
commercial_df['4_not_is_request'] = commercial_df.apply(not_is_request, axis=1)

# Analyze refined breakdown
txn_counts_4 = commercial_df['4_not_is_request'].value_counts()
print("\nRefined classification after Request exclusion (Likely Transaction vs Not):")
print(txn_counts_4)
print(f"Percentage classified as actual transaction: {(txn_counts_4.get(True, 0) / len(commercial_df)) * 100:.2f}%\n")

# Sample match
print("--- SAMPLE REFINED TRANSACTIONS NON-REQUEST (True) ---")
refined_true = commercial_df[commercial_df['4_not_is_request'] == True]['text'].dropna().sample(min(5, txn_counts_4.get(True, 0)), random_state=42)
for text in refined_true:
    print(f"- {text[:150].replace(chr(10), ' ')}")

# Sample unmatch
print("\n--- SAMPLE FILTERED OUT AS REQUEST (Passed Stage 3, but rejected in Stage 4) ---")
filtered_out = commercial_df[(commercial_df['3_not_is_promo'] == True) & (commercial_df['4_not_is_request'] == False)]['text'].dropna()
if len(filtered_out) > 0:
    for text in filtered_out.sample(min(5, len(filtered_out)), random_state=42):
        print(f"- {text[:150].replace(chr(10), ' ')}")
else:
    print("No messages were filtered out in this stage.")

STAGE 4: Applying requested exclusion filter on remaining messages...

Refined classification after Request exclusion (Likely Transaction vs Not):
4_not_is_request
False    8300
True     3359
Name: count, dtype: int64
Percentage classified as actual transaction: 28.81%

--- SAMPLE REFINED TRANSACTIONS NON-REQUEST (True) ---
- Rs600.0 debited@SBI UPI frm A/cX6254 on 26Dec22 RefNo 236049840363. If not done by u, fwd this SMS to 9223008333/Call 1800111109 or 09449112211 to blo
- Your Pay Later bill of Rs 1650.0 will be debited on 5th of this month from registered bank a/c. View Bill https://u.axio.co/h6hJrP0h5k2YQcr -axio
- PIZZA HUT's ultimate weekend treat that you can't miss out! Enjoy 4 Flavor Fun Pizzas with a Pepsi @ Rs.349. Order @ bit.ly/phofr T&C
- Binge all Night with Vi Chhota Hero pack! Recharge with Rs17 & get UNLIMITED Night Data from 12 am to 6 am for 1 night. Recharge NOW. bit.ly/Vi121NAT
- Unlimited Pack EXPIRY ALERT! Your Unlimited Pack is EXPIRING TOMORROW 1)Rs249:UL Ca

In [11]:
import os

# Separate the datasets based on Stage 4
likely_transactions = commercial_df[commercial_df['4_not_is_request'] == True]
non_transactions = commercial_df[commercial_df['4_not_is_request'] == False]

output_path = 'RESULTS/4_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 3359 likely transactions and 8300 non-transactions to RESULTS/4_transactions_split.xlsx


In [12]:
# ==========================================
# STAGE 5: FINANCIAL KEYWORD REQUIREMENT
# ==========================================
# Finally, ensure that the messages surviving so far actually contain action verbs 
# or specific context words that point to banking/financial events.

fin_pattern = re.compile(
    r'\b(debited|credited|deducted|received|spent|paid|payment|refunded|'
    r'account|acct|balance|bal|vpa|upi|neft|imps|rtgs|atm|pos|card|txn|transaction)\b|'
    r'a/c', 
    re.IGNORECASE
)

# NOTE: We removed `inr|rs|₹` from this list because Stage 1 ALREADY confirms the presence of money,
# we strictly want to check for the supporting action/entity context.

def has_financial_keyword_final(row):
    text = row['text']
    if not isinstance(text, str):
        return False
        
    # MUST have passed Stage 4 (Amount, No OTP, No Promo, No Request)
    if not row['4_not_is_request']:
        return False
        
    # MUST contain a financial action/entity keyword
    return bool(fin_pattern.search(text))

print("STAGE 5: Applying financial context keyword filter...")
commercial_df['5_has_fin_keyword'] = commercial_df.apply(has_financial_keyword_final, axis=1)

# Analyze refined breakdown
txn_counts_5 = commercial_df['5_has_fin_keyword'].value_counts()
print("\nRefined classification after Fin Keyword requirement (Likely Transaction vs Not):")
print(txn_counts_5)
print(f"Percentage classified as actual transaction: {(txn_counts_5.get(True, 0) / len(commercial_df)) * 100:.2f}%\n")

print("--- SAMPLE REFINED TRANSACTIONS FIN KEYWORD (True) ---")
refined_true = commercial_df[commercial_df['5_has_fin_keyword'] == True]['text'].dropna().sample(min(5, txn_counts_5.get(True, 0)), random_state=42)
for text in refined_true:
    print(f"- {text[:150].replace(chr(10), ' ')}")

print("\n--- SAMPLE FILTERED OUT LACKING FIN CONTEXT (Passed Stage 4, but rejected here) ---")
filtered_out = commercial_df[(commercial_df['4_not_is_request'] == True) & (commercial_df['5_has_fin_keyword'] == False)]['text'].dropna()
if len(filtered_out) > 0:
    for text in filtered_out.sample(min(5, len(filtered_out)), random_state=42):
        print(f"- {text[:150].replace(chr(10), ' ')}")
else:
    print("No messages were filtered out in this stage.")

STAGE 5: Applying financial context keyword filter...

Refined classification after Fin Keyword requirement (Likely Transaction vs Not):
5_has_fin_keyword
False    9797
True     1862
Name: count, dtype: int64
Percentage classified as actual transaction: 15.97%

--- SAMPLE REFINED TRANSACTIONS FIN KEYWORD (True) ---
- NEXTBILLION TECHNOLOGY at EOD 04/06/2022 reported your Fund bal Rs 35.070 & Securities bal 0. This excludes your Bank, DP and PMS bal with the broker-
- CDSL: Debit in a/c *08945296 for 299-DCW LTD - EQ RS 2 on 08FEB
- Rs.517.8 spent on HDFC Bank Card x6719 at IBIBOGROUPPVTLTD on 2024-03-13:15:45:18.Not U? To Block & Reissue Call 18002586161/SMS BLOCK CC 6719 to 7308
- Hi Mr Manish Satish Aradwad,  Love using UPI.  You will love this too.  Get Rs.250 voucher + more with HDFC Bank A/c.  Open Savings A/c now: hdfcbk.io
- Recharge of Rs. 209.0 is successful for your Jio number 8830529924. Entitlement: Benefits:  1. UNLIMITED DATA - 28 GB (1 GB/Day)  2. UNLIMITED CALLS  

--- 

In [13]:
import os

# Separate the datasets based on final Stage 5
likely_transactions = commercial_df[commercial_df['5_has_fin_keyword'] == True]
non_transactions = commercial_df[commercial_df['5_has_fin_keyword'] == False]

output_path = 'RESULTS/5_transactions_split.xlsx'
with pd.ExcelWriter(output_path) as writer:
    likely_transactions.to_excel(writer, sheet_name='likely_transactions', index=False)
    non_transactions.to_excel(writer, sheet_name='non_transactions', index=False)

print(f"Saved {len(likely_transactions)} likely transactions and {len(non_transactions)} non-transactions to {output_path}")

Saved 1862 likely transactions and 9797 non-transactions to RESULTS/5_transactions_split.xlsx
